# 12 - Historical Companies House snapshots (acquisition)

So far this project has run off a single Companies House snapshot (the June 2026 bulk file). That is enough to describe a company *today*, but it tells me nothing about how a company is *changing*, and change is the actual buying signal I care about (a jump in charges, a slide into distress, a step up in size). It also gives me nothing to predict, so there is no way to fit a model or run SHAP.

This notebook is step 1 of fixing that: I pull the **historical** bulk snapshots so I can build a monthly time series. Companies House only links the current month on its download page, but the older monthly zips stay on the server and are reachable by direct URL. I confirmed by probing that 33 months exist, Oct 2023 to Jul 2026.

One thing to flag up front: **June 2025 is genuinely missing** from the server. That is not a bug in my code, it is a real hole, and I handle it later when I compute deltas on a calendar-aware month spine (so a gap never silently corrupts a 3-month or 12-month difference).

I keep all the logic in `src/data/ch_bulk.py` and just drive it from here, the same way notebooks 10 and 11 lean on `src/features`.

## Setup

I point Python at the repo root so `from src.data import ch_bulk` resolves, exactly like the earlier notebooks do. The heavy libraries for the modelling steps (duckdb, lightgbm, shap, scikit-learn) are already installed in the project `.venv`; nothing new is needed just to download.

In [2]:
import sys
from pathlib import Path

# Make the repo root importable so `src` is visible (src is a namespace package).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.data import ch_bulk

print("repo root:", REPO_ROOT)
print("snapshots dir:", (REPO_ROOT / ch_bulk.SNAPSHOT_DIR).resolve())
print("manifest months:", len(ch_bulk.MANIFEST))

repo root: /home/viklin/repos/lloyds-commercial-banking-intelligence-2026
snapshots dir: /home/viklin/repos/lloyds-commercial-banking-intelligence-2026/data/raw/snapshots
manifest months: 33


## The manifest

`ch_bulk.MANIFEST` is the verified list of 33 snapshot dates. The filenames are not reliably the 1st of the month (some months land on the 4th, 7th, 2nd), so I probed days 01 to 15 of each month once and hard-coded the dates that actually resolved. Day to day I do not want to re-hit the server just to know the dates, so the module keeps the verified list and only probes on demand.

The cell below shows the dates and makes the June 2025 gap visible.

In [ ]:
import calendar

have_months = {d[:7] for d in ch_bulk.MANIFEST}
# Walk every month from Oct 2023 to Jul 2026 and mark which ones we have.
y, m = 2023, 10
while (y, m) <= (2026, 7):
    ym = f"{y}-{m:02d}"
    date = next((d for d in ch_bulk.MANIFEST if d.startswith(ym)), None)
    print(f"{ym}  {'-> ' + date if date else 'MISSING (real hole on CH server)'}")
    m += 1
    if m > 12:
        y, m = y + 1, 1

## (Optional) re-probe the server

If I ever want to refresh the manifest (for example a new month has published, or I am checking whether an older file has aged out), I can re-run the probe. It sends a cheap HEAD request per candidate day and takes the first hit per month. I leave this off by default so the notebook does not spam the CH server on every run.

In [ ]:
REPROBE = False
if REPROBE:
    months = [f"{d[:7]}" for d in ch_bulk.MANIFEST]  # or any list of 'YYYY-MM'
    live = ch_bulk.probe_snapshot_dates(months)
    print(f"{len(live)} live dates found")
    for d in live:
        print(" ", d)
else:
    print("Re-probe skipped; using the verified manifest.")

## Download

Now the actual pull. Notes on how this behaves:

- **Where:** everything goes to `data/raw/snapshots/`. That sits under `data/`, which is fully gitignored, so these ~16 GB of zips never get pushed to the online repo.
- **Resumable:** any zip already fully on disk is skipped, so I can re-run this cell freely and it only fetches what is missing. Each file is streamed to a `.part` and only renamed on success, so an interrupted download never leaves a half-file masquerading as complete.
- **Parallel:** 4 downloads at a time, which is a reasonable balance between speed and being polite to the server.
- **Retries:** each file gets up to 3 attempts before it is reported as FAILED.

I keep the zips permanently. Once Companies House ages these months off the server they are the only copy I have, and re-deriving the panel with a different column set later then costs zero downloads.

In [ ]:
# This is the ~16 GB pull. Safe to re-run: it skips anything already downloaded.
results = ch_bulk.download_snapshots(dest_dir=REPO_ROOT / ch_bulk.SNAPSHOT_DIR)

ok = sum(1 for s in results.values() if s.startswith(("downloaded", "skipped")))
print(f"\n{ok}/{len(results)} snapshots present.")
failed = {d: s for d, s in results.items() if s.startswith("FAILED")}
if failed:
    print("Failures (re-run the cell to retry):")
    for d, s in failed.items():
        print(" ", d, s)

## Verify what I have

A quick tally of which manifest dates are on disk, so I can confirm the acquisition is complete before moving on to building the panel in the next step.

In [3]:
status = ch_bulk.snapshot_status(dest_dir=REPO_ROOT / ch_bulk.SNAPSHOT_DIR)
present = [d for d, ok in status.items() if ok]
missing = [d for d, ok in status.items() if not ok]
print(f"present: {len(present)}/{len(status)}")
if missing:
    print("still missing:", missing)

total_gb = sum(
    (REPO_ROOT / ch_bulk.SNAPSHOT_DIR / f"BasicCompanyDataAsOneFile-{d}.zip").stat().st_size
    for d in present
) / (1 << 30)
print(f"total on disk: {total_gb:,.1f} GB")

present: 33/33
total on disk: 14.9 GB


## Next

With the 33 zips downloaded and kept, step 2 (still in this notebook, next section, once `src/features/panel.py` is in place) is to build the two-pass union panel: work out the sector universe across every month, then extract slim per-month rows for every universe member whatever its status, and write one parquet partition per snapshot. The headline check there is parity: the June 2026 partition, filtered back down to Active and in-sector, must exactly reproduce `filtered_bb_sme_sectors.csv` (1,372,321 rows), which proves the historical replay is faithful to notebook 1.